## Environment Setup

### Purpose
Loads environment variables and imports all required libraries for the RAG chat system.

### Key Actions
- Imports `os`, `json`, `re` for system operations and parsing.
- Loads environment variables using `dotenv`.
- Imports Pinecone client for vector retrieval.
- Imports LangChain wrappers:
  - `OpenAIEmbeddings` for generating embeddings from user queries.
  - `ChatOpenAI` for generating LLM responses.

### Result
The environment is fully prepared for initializing the RAG pipeline components.


In [ ]:
import os
import json
import re
from dotenv import load_dotenv

from pinecone import Pinecone
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

load_dotenv()

## Load API Keys & Pipeline State

### Purpose
Loads all required API keys and retrieves the active video namespace from the pipeline state file.

### Key Actions
- Reads environment variables:
  - `OPENAI_API_KEY`
  - `PINECONE_API_KEY`
  - `PINECONE_INDEX_NAME`
- Defines the path to `session_state.json`.
- Validates that the state file exists.
- Loads the JSON content and extracts `active_video_id`.
- Prints the active namespace, which determines where Pinecone will search for vectors.

### Result
The system now knows:
- Which API keys to use
- Which Pinecone index to query
- Which video namespace contains the embeddings for RAG


In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")

STATE_PATH = "data/session_state.json"

if not os.path.exists(STATE_PATH):
    raise FileNotFoundError("session_state.json not found")

with open(STATE_PATH, "r", encoding="utf-8") as f:
    session_state = json.load(f)

ACTIVE_VIDEO_ID = session_state["active_video_id"]

print("Active namespace:", ACTIVE_VIDEO_ID)

Active namespace: 4_oufjP2yeM


## Initialize Pinecone, Embeddings & LLM

### Purpose
Sets up all core components required for Retrieval‑Augmented Generation (RAG):
- Pinecone index connection  
- Embedding model for query vectorization  
- Chat model for generating final answers  

### Key Actions
- Initializes the Pinecone client using the API key.
- Connects to the configured Pinecone index (`PINECONE_INDEX_NAME`).
- Creates an embedding generator using `text-embedding-3-small`.
- Creates a chat model (`gpt-4o-mini`) with temperature 0 for deterministic answers.

### Result
The system is now ready to:
1. Convert user questions into embeddings  
2. Retrieve relevant chunks from Pinecone  
3. Generate grounded answers using the LLM  


In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## In‑Memory Chat State Manager

### Purpose
Implements a lightweight in‑memory store to maintain chat history per session.  
This allows the RAG chat system to remember previous user and assistant messages during a conversation.

### Key Components

#### `store`
A global dictionary acting as an in‑memory database.  
Each `session_id` maps to a structure containing:
- `chat_history`: list of past messages

#### `get_state(session_id)`
Retrieves the state for a given session.  
If the session does not exist yet, it initializes:
```json
{
  "chat_history": []
}


In [ ]:
store = {}

def get_state(session_id):
    if session_id not in store:
        store[session_id] = {
            "chat_history": []
        }
    return store[session_id]


def add_message(session_id, role, message):
    state = get_state(session_id)
    state["chat_history"].append({
        "role": role,
        "message": message
    })

## Question Rewriting Engine

### Purpose
Creates a mechanism to rewrite user questions into standalone queries that can be safely embedded and sent to Pinecone.  
This ensures the retrieval step works even when the user asks follow‑up questions that depend on chat history.

### Components

#### `chat_llm`
Initializes a deterministic chat model (`gpt-4o-mini`, temperature 0) used exclusively for rewriting questions.

---

### `format_history(chat_history)`
Converts the chat history into a readable text block.

Each message becomes:


In [ ]:
chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def format_history(chat_history):
    return "\n".join(
        [f"{m['role']}: {m['message']}" for m in chat_history]
    )


def rewrite_question(question, chat_history):
    if not chat_history:
        return question

    history_text = format_history(chat_history)

    prompt = f"""
Rewrite the user question into a standalone query.

Rules:
- Keep the meaning of the question
- Use chat history only for context
- Do NOT answer the question
- Output ONLY the rewritten query

Chat history:
{history_text}

Question:
{question}

Standalone query:
"""

    return chat_llm.invoke(prompt).content.strip()

## Retrieval Function (Pinecone Query)

### Purpose
Implements the core retrieval step of the RAG pipeline.  
Given a rewritten user query, this function:
1. Converts the query into an embedding  
2. Searches Pinecone for the most relevant chunks  
3. Returns metadata‑rich results for grounding the LLM answer  

### Components

#### `retrieve(query: str, top_k: int = 4)`
Executes semantic search over the active video namespace.

##### Steps:
- Generates an embedding for the query using `embeddings.embed_query()`.
- Sends the embedding to Pinecone using:
  - `vector`: the query embedding  
  - `namespace`: the active video ID  
  - `top_k`: number of chunks to retrieve  
  - `include_metadata=True`: ensures timestamps and text are returned  

##### Output:
A Pinecone query response containing:
- `matches`: list of retrieved chunks  
- Each chunk includes:
  - `score` (similarity)
  - `metadata` (timestamp, text, video_id)

### Result
This function provides the contextual evidence needed for the LLM to generate grounded answers based strictly on the video content.


In [ ]:
def retrieve(query: str, top_k: int = 4):
    query_embedding = embeddings.embed_query(query)

    return index.query(
        vector=query_embedding,
        namespace=ACTIVE_VIDEO_ID,
        top_k=top_k,
        include_metadata=True
    )

## Build Context from Retrieved Chunks

### Purpose
Transforms the raw Pinecone retrieval results into a clean, readable context block that will be injected into the LLM prompt.

### Key Actions
- Validates that the retrieval response contains `"matches"`.
- Iterates through each match returned by Pinecone.
- Extracts:
  - `text` — the chunk content  
  - `timestamp` — the anchor timestamp for that chunk  
- Formats each chunk as:


#### when a timestamp is available.

### Output
Returns a single string containing all retrieved chunks, separated by blank lines.  
This formatted context is what the LLM will use to generate grounded answers based strictly on the video content.

### Result
Provides a clean, human‑readable context block ready to be inserted into the RAG prompt.



In [ ]:
def build_context(results):
    if not results or "matches" not in results:
        return ""

    chunks = []

    for match in results["matches"]:
        metadata = match.get("metadata", {})
        text = metadata.get("text", "").strip()
        timestamp = metadata.get("timestamp", "")

        if text:
            chunks.append(f"[{timestamp}] {text}" if timestamp else text)

    return "\n\n".join(chunks)

## Full Chat Function (RAG + Memory + Debug)

### Purpose
Implements the complete multi‑turn RAG chat pipeline.  
This function orchestrates every step required to answer a user question using:
- Conversation history  
- Rewritten standalone queries  
- Pinecone retrieval  
- Video context  
- A structured LLM prompt  
- Token usage debugging  
- Memory updates  

---

## Components

### Token Counter
Uses `tiktoken` to estimate token usage for:
- History  
- Context  
- User question  
- Full prompt  

This helps monitor prompt size and cost.

---

### `chat(session_id: str, question: str)`
Main entry point for the RAG chat system.

---

## Step‑by‑Step Breakdown

### **1. Load conversation state**
Retrieves the session’s chat history from the in‑memory store.  
Determines whether this is the first message.

### **2. Rewrite question**
Uses the rewriting engine to convert the user question into a standalone query suitable for retrieval.

### **3. Retrieve context**
Queries Pinecone using the rewritten query.  
Builds a clean context block from the retrieved chunks (including timestamps).

### **4. Save user message**
Adds the user’s message to the chat history.

### **5. Format history**
Converts the updated chat history into a readable text block.

### **6. Build the RAG prompt**
Constructs a structured prompt for the LLM with:
- System persona (ResuMazing)
- First‑message behavior
- Conversation history
- Retrieved video context
- User question

Guidelines ensure:
- Friendly tone  
- No hallucinations  
- Use of history + context  
- ATS/CV‑focused answers  

---

### **7. Generate response**
Calls the LLM (`gpt-4o-mini`) to produce the final grounded answer.

Uses `get_openai_callback()` to capture:
- Prompt tokens  
- Completion tokens  
- Total cost  

---

### **Debug Output**
Prints detailed diagnostic information:
- Timestamp  
- Session ID  
- First‑message flag  
- Token breakdown  
- OpenAI usage  
- Question  
- Answer  

This is extremely useful during development.

---

### **8. Save assistant response**
Stores the assistant’s answer in the chat history.

---

### **Return Value**
Returns the final answer string produced by the LLM.

---

## Result
This cell provides a complete, production‑ready RAG chat engine with:
- Multi‑turn memory  
- Query rewriting  
- Pinecone retrieval  
- Timestamp‑aware context  
- Structured prompting  
- Debugging and token accounting  
- Safe, grounded LLM responses  


In [ ]:
from datetime import datetime
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")  # Change if using another model

def count_tokens(text):
    return len(enc.encode(text))


def chat(session_id: str, question: str):

    # -----------------------
    # 1. Load conversation state
    # -----------------------
    state = get_state(session_id)
    chat_history = state["chat_history"]

    # Is this the first conversation?
    is_first_message = len(chat_history) == 0

    # -----------------------
    # 2. Rewrite question for retrieval
    # -----------------------
    standalone_query = rewrite_question(question, chat_history)

    # -----------------------
    # 3. Retrieve relevant video chunks
    # -----------------------
    results = retrieve(standalone_query)
    context = build_context(results)

    # -----------------------
    # 4. Save current user message
    # -----------------------
    add_message(session_id, "user", question)

    # Reload updated history (now includes the user's message)
    state = get_state(session_id)
    chat_history = state["chat_history"]

    # -----------------------
    # 5. Build formatted history
    # -----------------------
    history_text = format_history(chat_history)

    # -----------------------
    # 6. Build prompt
    # -----------------------
    prompt = f"""
You are ResuMazing, an AI assistant specialized in resume writing, CV optimization and ATS optimization.

Your goal is to help users improve their resumes through friendly, practical and natural conversations.

You have access to two knowledge sources:

1. Conversation History
- Use it to remember information the user has previously shared.
- This is the primary source for questions about the user or previous messages.

2. Video Context
- Use it to answer questions about resumes, ATS optimization and career advice.
- Base your answers only on the retrieved context.

Guidelines:
- If FIRST_MESSAGE is True, begin with: "Welcome to ResuMazing!"
- Sound like a friendly career coach, not an AI assistant.
- Keep answers conversational, concise and helpful.
- Use conversation history whenever it contains the answer.
- Use video context whenever the question is about resumes or ATS.
- If both sources are relevant, combine them naturally.
- Never invent information.
- If you don't know the answer, say so in a friendly way and naturally guide the conversation back to resumes, CVs or ATS optimization.
- Avoid repetitive phrases like "Feel free to ask" or "Let me know if you need anything else."

FIRST_MESSAGE:
{is_first_message}

Conversation History:
{history_text}

Video Context:
{context}

User Question:
{question}
"""

    # -----------------------
    # Prompt statistics
    # -----------------------
    history_tokens = count_tokens(history_text)
    context_tokens = count_tokens(context)
    question_tokens = count_tokens(question)
    estimated_prompt_tokens = count_tokens(prompt)

    timestamp = datetime.now().strftime("%H:%M:%S")

    from langchain_community.callbacks.manager import get_openai_callback

    # -----------------------
    # 7. Generate response
    # -----------------------
    with get_openai_callback() as cb:
        answer = llm.invoke(prompt).content.strip()

    # -----------------------
    # Debug information
    # -----------------------
    print("\n" + "=" * 70)
    print("CHAT DEBUG")
    print("=" * 70)

    print(f"Timestamp           : {timestamp}")
    print(f"Session ID          : {session_id}")
    print(f"First Message       : {is_first_message}")

    print("-" * 70)

    print("Estimated Prompt Breakdown")
    print(f"History Tokens      : {history_tokens}")
    print(f"Context Tokens      : {context_tokens}")
    print(f"Question Tokens     : {question_tokens}")
    print(f"Estimated Prompt    : {estimated_prompt_tokens}")

    print("-" * 70)

    print("OpenAI Token Usage")
    print(f"Prompt Tokens       : {cb.prompt_tokens}")
    print(f"Completion Tokens   : {cb.completion_tokens}")
    print(f"Total Tokens        : {cb.total_tokens}")
    print(f"Estimated Cost (USD): ${cb.total_cost:.6f}")

    print("-" * 70)

    print("QUESTION")
    print(question)

    print("-" * 70)

    print("ANSWER")
    print(answer)

    print("=" * 70 + "\n")

    print("1. Loading state...")

    state = get_state(session_id)
    chat_history = state["chat_history"]

    print("2. Rewriting question...")

    standalone_query = rewrite_question(question, chat_history)

    print("3. Retrieving context...")

    results = retrieve(standalone_query)

    print("4. Building context...")

    context = build_context(results)

    print("5. Saving user message...")

    add_message(session_id, "user", question)

    print("6. Building prompt...")

    ...

    print("7. Calling LLM...")

    with get_openai_callback() as cb:

      answer = llm.invoke(prompt).content.strip()

    print("8. Saving assistant...")

    add_message(session_id, "assistant", answer)

    print("9. Done.")  

    # -----------------------
    # 8. Save assistant response
    # -----------------------
    add_message(session_id, "assistant", answer)

    return answer

In [ ]:
session_id = "test2"

print(chat(session_id, "Hi there... My name is Marcos and I live in Madrid"))
print(chat(session_id, "What is my name?"))
print(chat(session_id, "Where do I live?"))
print(chat(session_id, "Give me 3 tips to improve my resume"))
print(chat(session_id, "Which one is the most important?"))


CHAT DEBUG
Timestamp           : 16:18:06
Session ID          : test2
First Message       : False
----------------------------------------------------------------------
Estimated Prompt Breakdown
History Tokens      : 1550
Context Tokens      : 696
Question Tokens     : 12
Estimated Prompt    : 2516
----------------------------------------------------------------------
OpenAI Token Usage
Prompt Tokens       : 2523
Completion Tokens   : 41
Total Tokens        : 2564
Estimated Cost (USD): $0.000403
----------------------------------------------------------------------
QUESTION
Hi there... My name is Marcos and I live in Madrid
----------------------------------------------------------------------
ANSWER
Welcome back, Marcos! It’s great to see you again. How can I assist you today with your resume or job search? If you have any specific questions or need tips, just let me know!

Welcome back, Marcos! It’s great to see you again. How can I assist you today with your resume or job search? 

## Full Chat Function (RAG + Memory + Debug)

### Purpose
Implements the complete multi‑turn RAG chat pipeline.  
This function orchestrates every step required to answer a user question using:
- Conversation history  
- Rewritten standalone queries  
- Pinecone retrieval  
- Video context  
- A structured LLM prompt  
- Token usage debugging  
- Memory updates  

---

## Components

### Token Counter
Uses `tiktoken` to estimate token usage for:
- History  
- Context  
- User question  
- Full prompt  

This helps monitor prompt size and cost.

---

### `chat(session_id: str, question: str)`
Main entry point for the RAG chat system.

---

## Step‑by‑Step Breakdown

### **1. Load conversation state**
Retrieves the session’s chat history from the in‑memory store.  
Determines whether this is the first message.

### **2. Rewrite question**
Uses the rewriting engine to convert the user question into a standalone query suitable for retrieval.

### **3. Retrieve context**
Queries Pinecone using the rewritten query.  
Builds a clean context block from the retrieved chunks (including timestamps).

### **4. Save user message**
Adds the user’s message to the chat history.

### **5. Format history**
Converts the updated chat history into a readable text block.

### **6. Build the RAG prompt**
Constructs a structured prompt for the LLM with:
- System persona (ResuMazing)
- First‑message behavior
- Conversation history
- Retrieved video context
- User question

Guidelines ensure:
- Friendly tone  
- No hallucinations  
- Use of history + context  
- ATS/CV‑focused answers  

---

### **7. Generate response**
Calls the LLM (`gpt-4o-mini`) to produce the final grounded answer.

Uses `get_openai_callback()` to capture:
- Prompt tokens  
- Completion tokens  
- Total cost  

---

### **Debug Output**
Prints detailed diagnostic information:
- Timestamp  
- Session ID  
- First‑message flag  
- Token breakdown  
- OpenAI usage  
- Question  
- Answer  

This is extremely useful during development.

---

### **8. Save assistant response**
Stores the assistant’s answer in the chat history.

---

### **Return Value**
Returns the final answer string produced by the LLM.

---

## Result
This cell provides a complete, production‑ready RAG chat engine with:
- Multi‑turn memory  
- Query rewriting  
- Pinecone retrieval  
- Timestamp‑aware context  
- Structured prompting  
- Debugging and token accounting  
- Safe, grounded LLM responses  
